In [2]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats



#1 Define project files

BASE_DIR = Path.cwd()

SIMULATION_PATH = (
    BASE_DIR
    / "phase10_synthetic_learner_environment_rerun.csv"
)

PRIMARY_POLICY_PATH = (
    BASE_DIR
    / "phase17_final_haptic_policy_rerun.csv"
)

WEIGHTED_POLICY_PATH = (
    BASE_DIR
    / "phase18_weighted_haptic_policy_rerun.csv"
)

WEIGHTED_QTABLE_PATH = (
    BASE_DIR
    / "phase18_weighted_qtable_rerun.csv"
)

SUMMARY_PATH = (
    BASE_DIR
    / "phase18_weighted_reward_summary_rerun.csv"
)

EPISODE_PATH = (
    BASE_DIR
    / "phase18_weighted_episode_results_rerun.csv"
)

TEST_PATH = (
    BASE_DIR
    / "phase18_weighted_reward_tests_rerun.csv"
)


print(
    "Simulation exists:",
    SIMULATION_PATH.exists()
)

print(
    "Primary policy exists:",
    PRIMARY_POLICY_PATH.exists()
)


if not SIMULATION_PATH.exists():
    raise FileNotFoundError(
        SIMULATION_PATH
    )

if not PRIMARY_POLICY_PATH.exists():
    raise FileNotFoundError(
        PRIMARY_POLICY_PATH
    )



# 2 Load synthetic learner environment


simulation_df = pd.read_csv(
    SIMULATION_PATH
)

required_columns = [
    "Difficulty_D",
    "Cognitive_Effort_C",
    "D_level",
    "C_level",
    "Learner_Profile",
    "Learner_Ability",
    "Performance_P"
]

missing = [
    column
    for column in required_columns
    if column not in simulation_df.columns
]

if missing:
    raise ValueError(
        "Missing columns: "
        + ", ".join(missing)
    )


print(
    "\nSimulation shape:",
    simulation_df.shape
)



# 3 Define state and action spaces


STATES = [
    (d, c, p)
    for d in range(3)
    for c in range(3)
    for p in range(3)
]

ACTIONS = [
    0,
    1,
    2
]

HAPTIC_ACTIONS = {
    0: "Low",
    1: "Medium",
    2: "High"
}

BASE_COST = {
    0: 0.00,
    1: 0.03,
    2: 0.08
}


def performance_level(
    performance
):

    if performance < 1 / 3:
        return 0

    elif performance < 2 / 3:
        return 1

    else:
        return 2


def build_state(
    row,
    performance
):

    return (
        int(
            row["D_level"]
        ),
        int(
            row["C_level"]
        ),
        performance_level(
            performance
        )
    )



# 4 Define simulated learner-response model


def haptic_effect(
    action,
    difficulty,
    cognitive_effort,
    performance
):

    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (
            1 - performance
        )
    )

    if action == 0:
        return 0.00

    elif action == 1:
        return 0.15 * need

    elif action == 2:
        return 0.30 * need

    else:
        raise ValueError(
            "Action must be 0, 1, or 2."
        )


def probability_correct(
    ability,
    difficulty,
    cognitive_effort,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):

    score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    return (
        1
        / (
            1
            + np.exp(-score)
        )
    )


def probability_correct_with_haptics(
    ability,
    difficulty,
    cognitive_effort,
    performance,
    action,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):

    base_score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    effect = haptic_effect(
        action=action,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance
    )

    final_score = (
        base_score
        + effect
    )

    return (
        1
        / (
            1
            + np.exp(-final_score)
        )
    )



# 5 Define outcome- and history-weighted intervention cost


def weighted_haptic_cost(
    action,
    correct,
    long_term_performance
):
    """
    Extension experiment.

    If the learner response is unsuccessful,
    the full base intervention cost is applied.

    If the response is successful, stronger feedback
    is penalised less for learners with lower
    long-term performance and more for learners
    with stronger long-term performance.
    """

    base_cost = (
        BASE_COST[
            action
        ]
    )

    if action == 0:
        return 0.0

    long_term_performance = float(
        np.clip(
            long_term_performance,
            0.0,
            1.0
        )
    )

    if correct == 0:

        cost_multiplier = 1.0

    else:

        # Range approximately 0.50 to 1.00.
        # Lower-performing learners receive
        # a lighter intervention penalty.
        cost_multiplier = (
            0.50
            + 0.50
            * long_term_performance
        )

    return (
        base_cost
        * cost_multiplier
    )



# 6 Define primary and weighted reward functions


def primary_reward(
    correct,
    p_before,
    p_after,
    action
):

    success_reward = (
        1.0
        if correct == 1
        else -0.5
    )

    improvement = (
        p_after
        - p_before
    )

    return (
        success_reward
        + 2.0 * improvement
        - BASE_COST[
            action
        ]
    )


def weighted_reward(
    correct,
    p_before,
    p_after,
    action,
    long_term_performance
):

    success_reward = (
        1.0
        if correct == 1
        else -0.5
    )

    improvement = (
        p_after
        - p_before
    )

    cost = weighted_haptic_cost(
        action=action,
        correct=correct,
        long_term_performance=long_term_performance
    )

    return (
        success_reward
        + 2.0 * improvement
        - cost
    )



# 7 Check weighted-cost behaviour


print(
    "\nWEIGHTED COST SANITY CHECK"
)

for long_term_p in [
    0.20,
    0.50,
    0.80
]:

    print(
        "\nLong-term performance:",
        long_term_p
    )

    for action in ACTIONS:

        successful_cost = (
            weighted_haptic_cost(
                action=action,
                correct=1,
                long_term_performance=long_term_p
            )
        )

        unsuccessful_cost = (
            weighted_haptic_cost(
                action=action,
                correct=0,
                long_term_performance=long_term_p
            )
        )

        print(
            HAPTIC_ACTIONS[
                action
            ],
            "| successful cost =",
            round(
                successful_cost,
                4
            ),
            "| unsuccessful cost =",
            round(
                unsuccessful_cost,
                4
            )
        )



# 8 Define weighted environment interaction


def environment_step_weighted(
    row,
    performance,
    long_term_performance,
    action,
    rng
):

    ability = float(
        row[
            "Learner_Ability"
        ]
    )

    difficulty = float(
        row[
            "Difficulty_D"
        ]
    )

    cognitive_effort = float(
        row[
            "Cognitive_Effort_C"
        ]
    )

    p_before = probability_correct(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort
    )

    p_after = (
        probability_correct_with_haptics(
            ability=ability,
            difficulty=difficulty,
            cognitive_effort=cognitive_effort,
            performance=performance,
            action=action
        )
    )

    p_after = np.clip(
        p_after,
        0.0,
        1.0
    )

    correct = rng.binomial(
        n=1,
        p=p_after
    )

    reward = weighted_reward(
        correct=correct,
        p_before=p_before,
        p_after=p_after,
        action=action,
        long_term_performance=
            long_term_performance
    )

    new_performance = (
        0.8 * performance
        + 0.2 * correct
    )

    new_performance = np.clip(
        new_performance,
        0.0,
        1.0
    )

    return (
        new_performance,
        reward,
        correct,
        p_before,
        p_after
    )



# 9 Define epsilon-greedy action selection


def choose_action(
    state,
    Q,
    epsilon,
    rng
):

    if rng.random() < epsilon:

        return int(
            rng.choice(
                ACTIONS
            )
        )

    q_values = (
        Q[
            state
        ]
    )

    max_q = np.max(
        q_values
    )

    best_actions = np.flatnonzero(
        q_values
        == max_q
    )

    return int(
        rng.choice(
            best_actions
        )
    )



# 10 Train the needs-weighted Q-learning policy


LEARNING_RATE = 0.10

DISCOUNT_FACTOR = 0.95

EPSILON = 1.0

EPSILON_MIN = 0.05

EPSILON_DECAY = 0.995

N_EPISODES = 500

STEPS_PER_EPISODE = 50

TRAIN_SEED = 42


Q_WEIGHTED = {
    state:
        np.zeros(
            3
        )
    for state
    in STATES
}


rng = np.random.default_rng(
    TRAIN_SEED
)

epsilon = EPSILON


for episode in range(
    N_EPISODES
):

    row = simulation_df.iloc[
        rng.integers(
            0,
            len(
                simulation_df
            )
        )
    ]

    learner_profile = (
        row[
            "Learner_Profile"
        ]
    )

    performance = float(
        row[
            "Performance_P"
        ]
    )

    # Long-term history starts from
    # the initial Phase 10 performance.
    history_sum = (
        performance
    )

    history_count = 1

    long_term_performance = (
        performance
    )


    for step in range(
        STEPS_PER_EPISODE
    ):

        state = build_state(
            row,
            performance
        )

        action = choose_action(
            state=state,
            Q=Q_WEIGHTED,
            epsilon=epsilon,
            rng=rng
        )

        (
            new_performance,
            reward,
            correct,
            p_before,
            p_after
        ) = environment_step_weighted(
            row=row,
            performance=performance,
            long_term_performance=
                long_term_performance,
            action=action,
            rng=rng
        )

        # Update longer-term learner history
        history_sum += (
            correct
        )

        history_count += 1

        long_term_performance = (
            history_sum
            / history_count
        )


        learner_rows = (
            simulation_df[
                simulation_df[
                    "Learner_Profile"
                ]
                == learner_profile
            ]
        )

        next_row = learner_rows.iloc[
            rng.integers(
                0,
                len(
                    learner_rows
                )
            )
        ]

        next_state = build_state(
            next_row,
            new_performance
        )

        current_q = (
            Q_WEIGHTED[
                state
            ][
                action
            ]
        )

        best_next_q = np.max(
            Q_WEIGHTED[
                next_state
            ]
        )

        target = (
            reward
            + DISCOUNT_FACTOR
            * best_next_q
        )

        Q_WEIGHTED[
            state
        ][
            action
        ] = (
            current_q
            + LEARNING_RATE
            * (
                target
                - current_q
            )
        )

        performance = (
            new_performance
        )

        row = (
            next_row
        )


    epsilon = max(
        EPSILON_MIN,
        epsilon
        * EPSILON_DECAY
    )


print(
    "\nWeighted-policy training complete."
)

print(
    "Final epsilon:",
    round(
        epsilon,
        4
    )
)



# 11 Extract the weighted policy


WEIGHTED_POLICY = {}

for state in STATES:

    q_values = (
        Q_WEIGHTED[
            state
        ]
    )

    best_action = int(
        np.flatnonzero(
            q_values
            == np.max(
                q_values
            )
        )[0]
    )

    WEIGHTED_POLICY[
        state
    ] = (
        best_action
    )


print(
    "\nWeighted policy action counts:"
)

print(
    Counter(
        WEIGHTED_POLICY.values()
    )
)



# 12 Export weighted policy and Q-table


LEVEL_NAMES = {
    0: "Low",
    1: "Medium",
    2: "High"
}


weighted_policy_rows = []

weighted_q_rows = []


for state in STATES:

    d, c, p = (
        state
    )

    action = (
        WEIGHTED_POLICY[
            state
        ]
    )

    weighted_policy_rows.append(
        {
            "D_level":
                d,

            "D_label":
                LEVEL_NAMES[
                    d
                ],

            "C_level":
                c,

            "C_label":
                LEVEL_NAMES[
                    c
                ],

            "P_level":
                p,

            "P_label":
                LEVEL_NAMES[
                    p
                ],

            "Haptic_Action":
                action,

            "Haptic_Label":
                HAPTIC_ACTIONS[
                    action
                ]
        }
    )

    weighted_q_rows.append(
        {
            "D_level":
                d,

            "C_level":
                c,

            "P_level":
                p,

            "Q_Low":
                Q_WEIGHTED[
                    state
                ][0],

            "Q_Medium":
                Q_WEIGHTED[
                    state
                ][1],

            "Q_High":
                Q_WEIGHTED[
                    state
                ][2]
        }
    )


weighted_policy_df = pd.DataFrame(
    weighted_policy_rows
)

weighted_q_df = pd.DataFrame(
    weighted_q_rows
)


weighted_policy_df.to_csv(
    WEIGHTED_POLICY_PATH,
    index=False
)

weighted_q_df.to_csv(
    WEIGHTED_QTABLE_PATH,
    index=False
)



# 13 Load primary fixed-cost policy


primary_policy_df = pd.read_csv(
    PRIMARY_POLICY_PATH
)

PRIMARY_POLICY = {}

for _, row in primary_policy_df.iterrows():

    state = (
        int(
            row[
                "D_level"
            ]
        ),
        int(
            row[
                "C_level"
            ]
        ),
        int(
            row[
                "P_level"
            ]
        )
    )

    PRIMARY_POLICY[
        state
    ] = int(
        row[
            "Haptic_Action"
        ]
    )


print(
    "\nPrimary policy:",
    Counter(
        PRIMARY_POLICY.values()
    )
)

print(
    "Weighted policy:",
    Counter(
        WEIGHTED_POLICY.values()
    )
)



# 14 Compare policy decisions state by state


policy_comparison_rows = []

for state in STATES:

    primary_action = (
        PRIMARY_POLICY[
            state
        ]
    )

    weighted_action = (
        WEIGHTED_POLICY[
            state
        ]
    )

    policy_comparison_rows.append(
        {
            "State":
                str(
                    state
                ),

            "Primary_Action":
                primary_action,

            "Primary_Label":
                HAPTIC_ACTIONS[
                    primary_action
                ],

            "Weighted_Action":
                weighted_action,

            "Weighted_Label":
                HAPTIC_ACTIONS[
                    weighted_action
                ],

            "Changed":
                primary_action
                != weighted_action
        }
    )


policy_comparison_df = pd.DataFrame(
    policy_comparison_rows
)


print(
    "\nStates with changed recommendations:",
    int(
        policy_comparison_df[
            "Changed"
        ].sum()
    ),
    "/ 27"
)



# 15 Create paired evaluation conditions


N_TEST_EPISODES = 300
TEST_STEPS_PER_EPISODE = 50
TEST_SEED = 2026

setup_rng = np.random.default_rng(
    TEST_SEED
)

episode_start_indices = setup_rng.integers(
    0,
    len(simulation_df),
    size=N_TEST_EPISODES
)

episode_seeds = setup_rng.integers(
    0,
    2**32 - 1,
    size=N_TEST_EPISODES,
    dtype=np.uint32
)

print(
    "\nPaired evaluation conditions created."
)

print(
    "Episodes:",
    N_TEST_EPISODES
)

print(
    "Steps per episode:",
    TEST_STEPS_PER_EPISODE
)



# 16 Define primary fixed-cost environment step


def environment_step_primary(
    row,
    performance,
    action,
    rng
):

    ability = float(
        row[
            "Learner_Ability"
        ]
    )

    difficulty = float(
        row[
            "Difficulty_D"
        ]
    )

    cognitive_effort = float(
        row[
            "Cognitive_Effort_C"
        ]
    )

    p_before = probability_correct(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort
    )

    p_after = probability_correct_with_haptics(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance,
        action=action
    )

    p_after = np.clip(
        p_after,
        0.0,
        1.0
    )

    correct = rng.binomial(
        n=1,
        p=p_after
    )

    reward = primary_reward(
        correct=correct,
        p_before=p_before,
        p_after=p_after,
        action=action
    )

    new_performance = (
        0.8 * performance
        + 0.2 * correct
    )

    new_performance = np.clip(
        new_performance,
        0.0,
        1.0
    )

    return (
        new_performance,
        reward,
        correct,
        p_before,
        p_after
    )



# 17 Evaluate one frozen policy under paired conditions


def evaluate_policy_paired(
    policy_to_use,
    policy_name
):

    episode_rows = []

    decision_rows = []

    for episode_idx in range(
        N_TEST_EPISODES
    ):

        # Each policy receives the same episode seed.
        rng = np.random.default_rng(
            int(
                episode_seeds[
                    episode_idx
                ]
            )
        )

        # Each policy receives the same starting row.
        row = simulation_df.iloc[
            episode_start_indices[
                episode_idx
            ]
        ]

        learner_profile = (
            row[
                "Learner_Profile"
            ]
        )

        performance = float(
            row[
                "Performance_P"
            ]
        )

        # Longer-term performance history.
        history_sum = (
            performance
        )

        history_count = 1

        long_term_performance = (
            performance
        )

        total_primary_reward = 0.0
        total_weighted_reward = 0.0
        total_correct = 0

        for step in range(
            TEST_STEPS_PER_EPISODE
        ):

            state = build_state(
                row,
                performance
            )

            action = int(
                policy_to_use[
                    state
                ]
            )

            ability = float(
                row[
                    "Learner_Ability"
                ]
            )

            difficulty = float(
                row[
                    "Difficulty_D"
                ]
            )

            cognitive_effort = float(
                row[
                    "Cognitive_Effort_C"
                ]
            )

            p_before = probability_correct(
                ability=ability,
                difficulty=difficulty,
                cognitive_effort=cognitive_effort
            )

            p_after = (
                probability_correct_with_haptics(
                    ability=ability,
                    difficulty=difficulty,
                    cognitive_effort=cognitive_effort,
                    performance=performance,
                    action=action
                )
            )

            p_after = np.clip(
                p_after,
                0.0,
                1.0
            )

            # Same RNG structure is used for both policies.
            correct = rng.binomial(
                n=1,
                p=p_after
            )

            # Evaluate BOTH policies using BOTH reward definitions.
            reward_primary = primary_reward(
                correct=correct,
                p_before=p_before,
                p_after=p_after,
                action=action
            )

            reward_weighted = weighted_reward(
                correct=correct,
                p_before=p_before,
                p_after=p_after,
                action=action,
                long_term_performance=
                    long_term_performance
            )

            new_performance = (
                0.8 * performance
                + 0.2 * correct
            )

            new_performance = np.clip(
                new_performance,
                0.0,
                1.0
            )

            decision_rows.append(
                {
                    "Policy":
                        policy_name,

                    "Episode":
                        episode_idx + 1,

                    "Step":
                        step + 1,

                    "Learner_Profile":
                        learner_profile,

                    "D_level":
                        state[0],

                    "C_level":
                        state[1],

                    "P_level":
                        state[2],

                    "Action":
                        action,

                    "Haptic_Label":
                        HAPTIC_ACTIONS[
                            action
                        ],

                    "Correct":
                        correct,

                    "P_before":
                        p_before,

                    "P_after":
                        p_after,

                    "Primary_Reward":
                        reward_primary,

                    "Weighted_Reward":
                        reward_weighted,

                    "Long_Term_Performance":
                        long_term_performance
                }
            )

            total_primary_reward += (
                reward_primary
            )

            total_weighted_reward += (
                reward_weighted
            )

            total_correct += (
                correct
            )

            # Update longer-term performance history.
            history_sum += (
                correct
            )

            history_count += 1

            long_term_performance = (
                history_sum
                / history_count
            )

            # Keep the next passage within the same learner profile.
            learner_rows = simulation_df[
                simulation_df[
                    "Learner_Profile"
                ]
                == learner_profile
            ]

            next_row = learner_rows.iloc[
                rng.integers(
                    0,
                    len(
                        learner_rows
                    )
                )
            ]

            performance = (
                new_performance
            )

            row = (
                next_row
            )

        episode_rows.append(
            {
                "Policy":
                    policy_name,

                "Episode":
                    episode_idx + 1,

                "Learner_Profile":
                    learner_profile,

                "Mean_Primary_Reward":
                    total_primary_reward
                    / TEST_STEPS_PER_EPISODE,

                "Mean_Weighted_Reward":
                    total_weighted_reward
                    / TEST_STEPS_PER_EPISODE,

                "Accuracy":
                    total_correct
                    / TEST_STEPS_PER_EPISODE
            }
        )

    return (
        pd.DataFrame(
            episode_rows
        ),
        pd.DataFrame(
            decision_rows
        )
    )



# 18 Evaluate primary and weighted policies

(
    primary_episode_df,
    primary_decision_df
) = evaluate_policy_paired(
    policy_to_use=PRIMARY_POLICY,
    policy_name="Primary fixed-cost"
)

(
    weighted_episode_df,
    weighted_decision_df
) = evaluate_policy_paired(
    policy_to_use=WEIGHTED_POLICY,
    policy_name="Needs-weighted"
)


all_episode_df = pd.concat(
    [
        primary_episode_df,
        weighted_episode_df
    ],
    ignore_index=True
)

all_decision_df = pd.concat(
    [
        primary_decision_df,
        weighted_decision_df
    ],
    ignore_index=True
)



# 19 Summarise paired evaluation

summary_df = (
    all_episode_df
    .groupby(
        "Policy",
        as_index=False
    )
    .agg(
        Mean_Primary_Reward=(
            "Mean_Primary_Reward",
            "mean"
        ),

        SD_Primary_Reward=(
            "Mean_Primary_Reward",
            "std"
        ),

        Mean_Weighted_Reward=(
            "Mean_Weighted_Reward",
            "mean"
        ),

        SD_Weighted_Reward=(
            "Mean_Weighted_Reward",
            "std"
        ),

        Mean_Accuracy=(
            "Accuracy",
            "mean"
        ),

        SD_Accuracy=(
            "Accuracy",
            "std"
        )
    )
)


print(
    "\nPHASE 18 POLICY COMPARISON"
)

print(
    "--------------------------"
)

print(
    summary_df.round(
        5
    )
)



# 20 Summarise action distributions

action_summary = (
    all_decision_df
    .groupby(
        [
            "Policy",
            "Haptic_Label"
        ]
    )
    .size()
    .reset_index(
        name="Count"
    )
)

action_summary[
    "Proportion"
] = (
    action_summary[
        "Count"
    ]
    / action_summary
    .groupby(
        "Policy"
    )[
        "Count"
    ]
    .transform(
        "sum"
    )
)


print(
    "\nACTION DISTRIBUTIONS"
)

print(
    "--------------------"
)

print(
    action_summary.round(
        4
    )
)


# 21 Define paired statistical comparison

def paired_test(
    primary_values,
    weighted_values,
    comparison_name
):

    primary_values = np.asarray(
        primary_values,
        dtype=float
    )

    weighted_values = np.asarray(
        weighted_values,
        dtype=float
    )

    difference = (
        weighted_values
        - primary_values
    )

    n = len(
        difference
    )

    mean_difference = (
        difference.mean()
    )

    sd_difference = (
        difference.std(
            ddof=1
        )
    )

    se_difference = (
        sd_difference
        / np.sqrt(
            n
        )
    )

    critical_value = (
        stats.t.ppf(
            0.975,
            df=n - 1
        )
    )

    ci_low = (
        mean_difference
        - critical_value
        * se_difference
    )

    ci_high = (
        mean_difference
        + critical_value
        * se_difference
    )

    t_stat, t_p = (
        stats.ttest_rel(
            weighted_values,
            primary_values
        )
    )

    try:

        wilcoxon_stat, wilcoxon_p = (
            stats.wilcoxon(
                weighted_values,
                primary_values
            )
        )

    except ValueError:

        wilcoxon_stat = (
            np.nan
        )

        wilcoxon_p = (
            np.nan
        )

    if sd_difference > 0:

        cohens_dz = (
            mean_difference
            / sd_difference
        )

    else:

        cohens_dz = (
            np.nan
        )

    return {
        "Comparison":
            comparison_name,

        "N":
            n,

        "Mean_Difference_Weighted_Minus_Primary":
            mean_difference,

        "CI_Low":
            ci_low,

        "CI_High":
            ci_high,

        "t_Statistic":
            t_stat,

        "t_p":
            t_p,

        "Wilcoxon_Statistic":
            wilcoxon_stat,

        "Wilcoxon_p":
            wilcoxon_p,

        "Cohens_dz":
            cohens_dz
    }



# 22 Run paired statistical tests


test_rows = []


test_rows.append(
    paired_test(
        primary_episode_df[
            "Mean_Primary_Reward"
        ],
        weighted_episode_df[
            "Mean_Primary_Reward"
        ],
        "Primary reward"
    )
)


test_rows.append(
    paired_test(
        primary_episode_df[
            "Mean_Weighted_Reward"
        ],
        weighted_episode_df[
            "Mean_Weighted_Reward"
        ],
        "Weighted reward"
    )
)


test_rows.append(
    paired_test(
        primary_episode_df[
            "Accuracy"
        ],
        weighted_episode_df[
            "Accuracy"
        ],
        "Accuracy"
    )
)


tests_df = pd.DataFrame(
    test_rows
)


print(
    "\nPAIRED POLICY TESTS"
)

print(
    "-------------------"
)

print(
    tests_df.round(
        5
    )
)



# 23 Compare results by learner profile


profile_summary_df = (
    all_episode_df
    .groupby(
        [
            "Learner_Profile",
            "Policy"
        ],
        as_index=False
    )
    .agg(
        Mean_Primary_Reward=(
            "Mean_Primary_Reward",
            "mean"
        ),

        Mean_Weighted_Reward=(
            "Mean_Weighted_Reward",
            "mean"
        ),

        Mean_Accuracy=(
            "Accuracy",
            "mean"
        ),

        Episodes=(
            "Episode",
            "count"
        )
    )
)


print(
    "\nRESULTS BY LEARNER PROFILE"
)

print(
    "--------------------------"
)

print(
    profile_summary_df.round(
        5
    )
)



# 24 Save complete Phase 18 outputs


POLICY_COMPARISON_PATH = (
    BASE_DIR
    / "phase18_policy_state_comparison_rerun.csv"
)

EVALUATION_SUMMARY_PATH = (
    BASE_DIR
    / "phase18_policy_evaluation_summary_rerun.csv"
)

EPISODE_RESULTS_PATH = (
    BASE_DIR
    / "phase18_policy_episode_results_rerun.csv"
)

DECISION_RESULTS_PATH = (
    BASE_DIR
    / "phase18_policy_decision_results_rerun.csv"
)

ACTION_SUMMARY_PATH = (
    BASE_DIR
    / "phase18_policy_action_distribution_rerun.csv"
)

STATISTICAL_TESTS_PATH = (
    BASE_DIR
    / "phase18_policy_statistical_tests_rerun.csv"
)

PROFILE_SUMMARY_PATH = (
    BASE_DIR
    / "phase18_policy_profile_results_rerun.csv"
)


policy_comparison_df.to_csv(
    POLICY_COMPARISON_PATH,
    index=False
)

summary_df.to_csv(
    EVALUATION_SUMMARY_PATH,
    index=False
)

all_episode_df.to_csv(
    EPISODE_RESULTS_PATH,
    index=False
)

all_decision_df.to_csv(
    DECISION_RESULTS_PATH,
    index=False
)

action_summary.to_csv(
    ACTION_SUMMARY_PATH,
    index=False
)

tests_df.to_csv(
    STATISTICAL_TESTS_PATH,
    index=False
)

profile_summary_df.to_csv(
    PROFILE_SUMMARY_PATH,
    index=False
)



# 25 Final Phase 18 checks


print(
    "\nPHASE 18 FINAL CHECK"
)

print(
    "--------------------"
)

print(
    "Primary policy states:",
    len(
        PRIMARY_POLICY
    )
)

print(
    "Weighted policy states:",
    len(
        WEIGHTED_POLICY
    )
)

print(
    "Changed states:",
    int(
        policy_comparison_df[
            "Changed"
        ].sum()
    ),
    "/ 27"
)

print(
    "Evaluation episodes per policy:",
    N_TEST_EPISODES
)

print(
    "Evaluation decisions per policy:",
    N_TEST_EPISODES
    * TEST_STEPS_PER_EPISODE
)

# 26 Confirm Phase 18 completion

print(
    "\nPHASE 18 COMPLETE"
)

print(
    "Needs-weighted reward extension "
    "evaluated against the primary policy."
)

print(
    "\nSaved weighted policy:",
    WEIGHTED_POLICY_PATH
)

print(
    "Saved weighted Q-table:",
    WEIGHTED_QTABLE_PATH
)

print(
    "Saved policy-state comparison:",
    POLICY_COMPARISON_PATH
)

print(
    "Saved evaluation summary:",
    EVALUATION_SUMMARY_PATH
)

print(
    "Saved statistical tests:",
    STATISTICAL_TESTS_PATH
)

print(
    "Saved learner-profile results:",
    PROFILE_SUMMARY_PATH
)

Simulation exists: True
Primary policy exists: True

Simulation shape: (2127, 37)

WEIGHTED COST SANITY CHECK

Long-term performance: 0.2
Low | successful cost = 0.0 | unsuccessful cost = 0.0
Medium | successful cost = 0.018 | unsuccessful cost = 0.03
High | successful cost = 0.048 | unsuccessful cost = 0.08

Long-term performance: 0.5
Low | successful cost = 0.0 | unsuccessful cost = 0.0
Medium | successful cost = 0.0225 | unsuccessful cost = 0.03
High | successful cost = 0.06 | unsuccessful cost = 0.08

Long-term performance: 0.8
Low | successful cost = 0.0 | unsuccessful cost = 0.0
Medium | successful cost = 0.027 | unsuccessful cost = 0.03
High | successful cost = 0.072 | unsuccessful cost = 0.08

Weighted-policy training complete.
Final epsilon: 0.0816

Weighted policy action counts:
Counter({1: 12, 2: 10, 0: 5})

Primary policy: Counter({0: 10, 2: 9, 1: 8})
Weighted policy: Counter({1: 12, 2: 10, 0: 5})

States with changed recommendations: 14 / 27

Paired evaluation conditions c